# How to solve a new problem using this codebase?

The codebase supports three different problems

* Turan problem: what's the maximum number of edges of an undirected graph with `N` nodes such that there is no cycle of length 4?
* No isosceles on a grid: what's the maximum number of points in a grid `N x N` so that no three of them form the vertices of a (possible flat) isosceles triangle?
* No 5 on a sphere problem: what's the maximum number of points in a grid `N x N x N` such that no 5 points lie on a sphere?

Of course you may be interested in other problems. This notebook guides you on how to have a new environment you can use for training. 

## How to create a new environment?

Here is a step-by-step guide to implement your own math problem in the pipeline.

1. Create a file for your environement in `src/envs/` (e.g. `my_env.py`) and use the following (minimal) imports:
```
from src.envs.environment import DataPoint, BaseEnvironment
from src.envs.tokenizers import SparseTokenizer, DenseTokenizer
from src.utils import bool_flag
```
You can also copy-paste one of the existing files (e.g. `isoceles.py`) and adapt from it.

1. In this file, define the class corresponding to the mathematical object you want to study. This should be a DataPoint class with:
   - `__init__`: Initialize data structure. 
   - `calc_score`: Compute objective value (-1 if invalid, e.g. when some constraints are not satisfied, larger or equal to 0 if valid, and the highest the best)
   - `calc_features`: (Optional) Create a 1 to 1 mapping between mathematical object and string for deduplication
   - `local_search`: provide an algorithm that can: (i) fix your DataPoint if invalid (ii) (optionally) try to modify it to improve its score.
   - `_batch_generate_and_score`: provide a method to generate random (valid) instances of the mathematical object


2. Create a Tokenizer for your environment. For this, there are two options:
  - Option 1: if your mathematical objects can be represented by a sequence of point with k-coordinates then you can skip this step and use one of the existing tokenizers in the file `tokenizers.py`.
  - Option 2: if not, or if you prefer to have your own tokenizer, you can create a class based on `Tokenizer` in the file `tokenizers.py`. It should contain:
    - A method `encode` that takes an instance of DataPoint and represent it (1-to-1) in a list of tokens
    - A method `decode` that takes a list of tokens and re-create the corresponding DataPoint

2. Define the Environment class itself, based on BaseEnvironment, and specify:
   - `data_class`: Reference to your DataPoint class. This should be specified before the `__init__` of your new Environment class.
   - a tokenizer (see previous step) in the `__init__` of your new Environment  class. If you went for Option 1 you can use one of the classes already implemented in `tokenizers.py` and you just have to additionnally specify before the `__init__` of your new Environment class
      - `k`: Dimension of each element's index (e.g. a graph has `k=2` since it can be represented by a sequence of two-dimensional points each representing an edge)
      - `are_coordinates_symmetric`: Whether indices can be permuted in elements of the sequence (e.g. this is `True` for an undirected graph)
   - A method `register_args` which specifies command-line arguments corresponding to additional parameters specific to your mathematical environment (e.g. different possible generation methods, largest integer used, etc.), as well as their default values.

3. Register your environement in `envs/__init__.py` by adding it to the dictionnary `ENVS`. (e.g. `MyEnv:"my_env"`).

You can know use this pipeline on this problem by setting `env_name` to the key you provided in the dictionnary `ENVS` (e.g. `my_env`).



### Example 1: No 3-Term Arithmetic Progression in {0, ..., `N`-1}


Given the set {0, 1, 2, ..., N-1}, find the largest subset such that no three elements form an arithmetic progression.

#### Step 1: Build a DataPoint structure. 

DataPoint is responsible to store the example information in `self.data`. In addition, DataPoint has the method to run `local_search()`, therefore improve the current data point. 

In [ ]:
import numpy as np
from itertools import combinations
from src.envs.environment import DataPoint


class ArithmeticProgressionFreeDataPoint(DataPoint): # <- New class representing the mathematical object, based on DataPoint
    """
    Represents an Arithmetic Progression-free subset of {0, 1, ..., N-1}.
    
    Attributes:
        N: Universe size {0, 1, ..., N-1}
        data: Binary array, data[i] = 1 means element i is selected
        progressions: List of arithmetic progressions (violations, needed for local search)
        score: Number of elements if valid, -1 otherwise
        features: String representation
    """
    
    def __init__(self, N, init=False):
        self.N = N
        self.data = np.zeros(N, dtype=np.uint8)
        self.progressions = []        
        if init: # <- Required by the pipeline; If true, we want to create an element at random. This will be used during the process to differenciate between generation and decoding of the output of the neural network.
            self._add_elements_greedily() # <- Method to create a valid construction at random.
            self.calc_features() 
            self.calc_score()
    
    def get_elements(self): # <- Method specific to the problem.
        """
        Return set of selected elements. 
        """
        return set([i for i in range(self.N) if self.data[i] == 1])
    
    def calc_score(self): # <- Method required by the pipeline
        """
        Calculate the score.
        Score = number of elements if no arithmetic progression exists, -1 otherwise.
        """
        if len(self.progressions) > 0:
            self.score = -1
        else:
            self.score = int(self.data.sum())
    
    def calc_features(self): # <- Optional method used by the pipeline
        """
        Create a binary string representation for deduplication.
        """
        self.features = ",".join(str(self.data[i]) for i in range(self.N))
    
    def _would_create_ap(self, new_element, existing_set): # <- Method specific to the problem.
        """
        Check if adding new_element would create an arithmetic progression.
        """
        # Case 1: new_element is the middle (b)
        # We need a, c such that a + c = 2 * new_element
        target = 2 * new_element
        for a in existing_set:
            c = target - a
            if c != a and c in existing_set:
                # Found a, c such that a + c = 2 * new_element
                return True
        
        # Case 2: new_element is an endpoint
        for b in existing_set:
            c = 2 * b - new_element
            if c != new_element and c != b and c in existing_set:
                return True
            
        return False
    
    def _add_elements_greedily(self): # <- Method specific to the problem.
        """
        Greedily add elements while avoiding arithmetic progressions.
        """
        order = list(range(self.N))
        np.random.shuffle(order)
        
        current_set = self.get_elements()
        
        for elem in order:
            if not self._would_create_ap(elem, current_set):
                self.data[elem] = 1
                current_set.add(elem)
    
    def _compute_progressions(self): # <- Method specific to the problem.
        """
        Find all arithmetic progressions in the current selection.
        """
        elements = self.get_elements()
        element_set = set(elements)
        self.progressions = []
        
        for a, c in combinations(elements, 2):
            if (a + c) % 2 == 0:
                b = (a + c) // 2
                if b in element_set and a < b < c:
                    self.progressions.append((a, b, c))
    
    def _remove_elements_greedily(self): # <- Method specific to the problem.
        """
        Greedily remove elements to eliminate all arithmetic progressions.
        """
        while self.progressions:
            # Count how many arithmetic progressions each element appears in
            element_count = {}
            for ap in self.progressions:
                for elem in ap:
                    element_count[elem] = element_count.get(elem, 0) + 1
            
            # Remove the element in the most arithmetic progressions
            worst_element = max(element_count, key=element_count.get)
            self.data[worst_element] = 0
            
            # Update progressions
            self.progressions = [ap for ap in self.progressions if worst_element not in ap]
    
    def local_search(self, improve_with_local_search=True): # <- Method required by the pipeline
        """
        Apply local search to fix violations and optionally improve.
        """
        # Step 1: Find all violations
        self._compute_progressions()
        
        # Step 2: Remove elements to fix violations
        self._remove_elements_greedily()
        
        # Step 3: Optionally add more elements
        if improve_with_local_search:
            self._add_elements_greedily()
        
        # Step 4: Recompute progressions (this step is not strictly needed but it's done for consistency)
        self._compute_progressions()
        self.calc_features()
        self.calc_score()
    
    @classmethod
    def _update_class_params(cls, pars): # <- Method required by the pipeline
        pass
    
    @classmethod
    def _save_class_params(cls): # <- Method required by the pipeline
        pass

#### Step 2: Test this DataPoint structure.

You will save a lot of time if you test the DataPoint structure before launching any training.

In [4]:
# random data point with N=50
dp1 = ArithmeticProgressionFreeDataPoint(N=50, init=True)

print(f"Universe size: {dp1.N}")
print(f"Number of elements: {dp1.score}")
print(f"Elements: {dp1.get_elements()}")
print(f"Arithmetic progressions: {dp1.progressions}")

elements = dp1.get_elements()
for a, b, c in combinations(elements, 3):
    if 2 * b == a + c:
        print(f"ERROR: Found AP ({a}, {b}, {c})")
        break
else:
    print("Verified: No arithmetic progressions!\n")


# a datapoint with some violations
dp2 = ArithmeticProgressionFreeDataPoint(N=50, init=False)
# in this way, I force the creation of a point with violations
dp2.data[0] = 1
dp2.data[1] = 1
dp2.data[2] = 1

dp2._compute_progressions()
print(f"Arithmetic progressions before running local search: {dp2.progressions}")

dp2.local_search(improve_with_local_search=False)
print(f"Arithmetic progressions after running local search but without local improvement: {dp2.progressions}")
print(f"Score after running local search but without local improvement: {dp2.score}")

dp2.local_search(improve_with_local_search=True)
print(f"Arithmetic progressions after running local search and local improvement: {dp2.progressions}")
print(f"Score after running local search and local improvement: {dp2.score}")

Universe size: 50
Number of elements: 14
Elements: {32, 1, 34, 7, 41, 10, 44, 14, 15, 46, 17, 25, 26, 31}
Arithmetic progressions: []
Verified: No arithmetic progressions!

Arithmetic progressions before running local search: [(0, 1, 2)]
Arithmetic progressions after running local search but without local improvement: []
Score after running local search but without local improvement: 2
Arithmetic progressions after running local search and local improvement: []
Score after running local search and local improvement: 13


#### Step 3: Create the Environment.

The Environment is the link between the DataPoint and the tokenizer. Tokenizer is the tool to convert the mathematical object into tokens readable from Decoder-only models.

In [ ]:
from src.envs.environment import BaseEnvironment
from src.envs.tokenizers import SparseTokenizerSingleInteger


class ArithmeticProgressionFreeEnvironment(BaseEnvironment):
    # this problem lives in N^1, therefore k=1
    # for k=1, are_coordinates_symmetric is not applicable
    # Here we opt for Option 1 and we use an already implemented tokenizer 

    k = 1 # <- has to be specified with Option 1
    are_coordinates_symmetric = False # <- has to be specified with Option 1
    data_class = ArithmeticProgressionFreeDataPoint # <- has to be specified
    
    def __init__(self, params):
        super().__init__(params)
        self.tokenizer = SparseTokenizerSingleInteger(self.data_class, params.N, self.k, self.are_coordinates_symmetric, self.SPECIAL_SYMBOLS) # <- Our choice of pre-implemented tokenizer 
    
    @staticmethod
    def register_args(parser):
        """
        Register environment parameters.
        """
        parser.add_argument("--N", type=int, default=100, help="Universe size {0, 1, ..., N-1}") #<- Here we only add one parameter specific to the problem, but there could be many others :)


#### Step 4: Register the Environment

In order to have it runnable from the main program, you need to add a new line in the `src/envs/__init__.py`

```
ENVS = {
    "square": SquareEnvironment,
    "isosceles": IsoscelesEnvironment,
    "sphere": SphereEnvironment,
    "arithmetic_progression": ArithmeticProgressionFreeEnvironment,  # <- Add this line
}
```

### Example 2: No 3 collinear points on a `N x N` grid

Given an `N x N` grid, find the maximum number of points you can place such that no three points lie on the same straight line.

In this example, I will introduce two additional ideas:
- The problem is invariant under rotations and reflections of the grid, so we can deduplicate grids up to the 8 symmetries of the square. `make_object_canonical` controls whether we remove duplicates that differ only by these symmetries.
- For a given grid, there are 8 possible ways to convert it to its tokenized version. `augment_data_representation` controls if we want to augment data with these 8 symmetries.

#### Step 1: Add two specific functions to make object canonical and augment data

In [6]:
import numpy as np
import random

# We pick the lexicographically smallest of the 8 symmetries
def canonical_form_2d(matrix):
    best = None
    best_matrix = None

    # 8 symmetries: 4 rotations × 2 (with/without reflection)
    current = matrix
    for _ in range(4):
        flat = current.flatten().tolist()
        if best is None or flat < best:
            best = flat
            best_matrix = current.copy()

        reflected = np.flip(current, axis=1)
        flat = reflected.flatten().tolist()
        if flat < best:
            best = flat
            best_matrix = reflected.copy()

        current = np.rot90(current)

    return best_matrix


# We pick one of the 8 symmetries at random
def random_symmetry_2d(matrix):
    k = random.randint(0, 3)
    result = np.rot90(matrix, k)

    if random.randint(0, 1):
        result = np.flip(result, axis=1)

    return result.copy()


#### Step 2: Build a DataPoint structure. 

In [ ]:
import numpy as np
from itertools import combinations
from src.envs.environment import DataPoint


class CollinearDataPoint(DataPoint):
    """
    Represents a set of points on an N x N grid with no three collinear.
    
    Attributes:
        N: Grid size
        data: N x N binary matrix, data[i,j] = 1 means point (i,j) is selected
        collinear_triplets: List of collinear triplets (violations, needed for local search)
        score: Number of points if valid, -1 otherwise
        features: String representation
    """
    
    MAKE_OBJECT_CANONICAL = False # <- Additional argument specific to the problem
    
    def __init__(self, N, init=False):
        self.N = N
        self.data = np.zeros((N, N), dtype=np.uint8)
        self.collinear_triplets = []
        
        if init: # <- Required by the pipeline; If true, we want to create an element at random. This will be used during the process to differenciate between generation and decoding of the output of the neural network.
            self._add_points_greedily() # <- Method to create a valid construction at random.
            if self.MAKE_OBJECT_CANONICAL:
                self.data = canonical_form_2d(self.data)
            self.calc_features() # <- Required by the pipeline;
            self.calc_score() # <- Required by the pipeline;
    
    def get_points(self): # <- Method specific to the problem
        """Return list of selected points as (x, y) tuples."""
        return [(i, j) for i in range(self.N) for j in range(self.N) if self.data[i, j] == 1]
    
    def calc_score(self): # <- Method required by the pipeline
        if len(self.collinear_triplets) > 0:
            self.score = -1
        else:
            self.score = int(self.data.sum())
    
    def calc_features(self): # <- Optional method used by the pipeline
        """
        Create a string representation of the configuration for deduplication.
        """
        w = []
        for i in range(self.N):
            for j in range(self.N):
                w.append(self.data[i, j])
        self.features = ",".join(map(str, w))
    
    def _are_collinear(self, p1, p2, p3): # <- Method specific to the problem
        """Check if three points are collinear using cross product."""
        x1, y1 = p1
        x2, y2 = p2
        x3, y3 = p3
        cross = (x2 - x1) * (y3 - y1) - (x3 - x1) * (y2 - y1)
        return cross == 0
    
    def _would_create_collinear(self, new_point, existing_points): # <- Method specific to the problem
        """
        Check if adding new_point would create a collinear triplet with any pair of existing points.
        """
        for p1, p2 in combinations(existing_points, 2):
            if self._are_collinear(p1, p2, new_point):
                return True
        return False
    
    def _add_points_greedily(self): # <- Method specific to the problem
        """
        Greedily add points to the grid while avoiding collinear triplets.
        We iterate through all grid positions in random order and add a point if it doesn't create a collinear triplet.
        """
        all_positions = [(i, j) for i in range(self.N) for j in range(self.N)]
        np.random.shuffle(all_positions)
        
        current_points = self.get_points()
        
        for pos in all_positions:
            if self.data[pos] == 0:
                if not self._would_create_collinear(pos, current_points):
                    self.data[pos] = 1
                    current_points.append(pos)
    
    def _compute_collinear_triplets(self): # <- Method specific to the problem
        """Find all collinear triplets in the current configuration."""
        current_points = self.get_points()
        self.collinear_triplets = []
        
        for p1, p2, p3 in combinations(current_points, 3):
            if self._are_collinear(p1, p2, p3):
                triplet = tuple(sorted([p1, p2, p3]))
                self.collinear_triplets.append(triplet)
    
    def _remove_points_greedily(self): # <- Method specific to the problem
        """
        Greedily remove points to eliminate all collinear triplets.
        At each step, remove the point that appears in the most triplets.
        """
        while self.collinear_triplets:
            # Count how many triplets each point appears in
            point_count = {}
            for triplet in self.collinear_triplets:
                for point in triplet:
                    point_count[point] = point_count.get(point, 0) + 1
            
            # Find the point in the most triplets
            worst_point = max(point_count, key=point_count.get)
            
            # Remove it
            self.data[worst_point] = 0
            
            # Update triplets (remove all triplets containing this point)
            self.collinear_triplets = [t for t in self.collinear_triplets if worst_point not in t]
    
    def local_search(self, improve_with_local_search=True): # <- Method requiremd by the pipeline
        """
        Apply local search to fix violations and optionally improve the solution.
        """
        # Step 1: Find all violations
        self._compute_collinear_triplets()
        
        # Step 2: Remove points to fix violations
        self._remove_points_greedily()
        
        # Step 3: Optionally try to add more points
        if improve_with_local_search:
            self._add_points_greedily()
        
        # Step 4: Recompute progressions (this step is not strictly needed but it's done for consistency)
        self._compute_collinear_triplets()
        self.calc_features()
        self.calc_score()
    
    @classmethod
    def _update_class_params(cls, pars): # <- Method requiremd by the pipeline
        """Update class-level parameters (used for multiprocessing)."""
        cls.MAKE_OBJECT_CANONICAL = pars
    
    @classmethod
    def _save_class_params(cls): # <- Method requiremd by the pipeline
        """Save class-level parameters (used for multiprocessing)."""
        return cls.MAKE_OBJECT_CANONICAL


#### Step 3: Test the new data point class

In [8]:
dp1 = CollinearDataPoint(N=10, init=True)

print(f"Grid size: {dp1.N} x {dp1.N}")
print(f"Number of points: {dp1.score}")
print(f"Points: {dp1.get_points()}")
print(f"Collinear triplets: {dp1.collinear_triplets}\n")


# a datapoint with some violations
dp2 = CollinearDataPoint(N=10, init=False)
# in this way, I force the creation of a point with violations
dp2.data[0, 0] = 1
dp2.data[1, 1] = 1
dp2.data[2, 2] = 1

dp2._compute_collinear_triplets()
print(f"Collinear triplets before running local search: {dp2.collinear_triplets}")

dp2.local_search(improve_with_local_search=False)
print(f"Collinear triplets after running local search but without local improvement: {dp2.collinear_triplets}")
print(f"Score after running local search but without local improvement: {dp2.score}")

dp2.local_search(improve_with_local_search=True)
print(f"Collinear triplets after running local search and local improvement: {dp2.collinear_triplets}")
print(f"Score after running local search and local improvement: {dp2.score}")

Grid size: 10 x 10
Number of points: 15
Points: [(0, 7), (0, 9), (1, 1), (1, 2), (3, 8), (4, 4), (4, 9), (5, 2), (6, 3), (6, 8), (7, 3), (7, 5), (8, 0), (8, 4), (9, 7)]
Collinear triplets: []

Collinear triplets before running local search: [((0, 0), (1, 1), (2, 2))]
Collinear triplets after running local search but without local improvement: []
Score after running local search but without local improvement: 2
Collinear triplets after running local search and local improvement: []
Score after running local search and local improvement: 16


#### Step 4: Create the Environment Class

In [ ]:
from src.envs.environment import BaseEnvironment
from src.envs.tokenizers import SparseTokenizerSingleInteger, SparseTokenizerSequenceKTokens, DenseTokenizer
from src.utils import bool_flag


class CollinearEnvironment(BaseEnvironment):
    # this problem lives in N^2, therefore k=2
    # (i, j) or (j, i) represents two distinct points in the grid, therefore are_coordinates_symmetric=False    
    # Here we opt for Option 1 and we use *several* already implemented tokenizers 

    k = 2 # <- has to be specified with Option 1
    are_coordinates_symmetric = False # <- has to be specified with Option 1
    data_class = CollinearDataPoint # <- has to be specified
    
    def __init__(self, params):
        super().__init__(params)
        self.data_class.MAKE_OBJECT_CANONICAL = params.make_object_canonical
        encoding_augmentation = random_symmetry_2d if params.augment_data_representation else None
        if params.encoding_tokens == "single_integer":
            self.tokenizer = SparseTokenizerSingleInteger(
                self.data_class, params.N, self.k, self.are_coordinates_symmetric, self.SPECIAL_SYMBOLS, encoding_augmentation=encoding_augmentation
            ) # <- Possible choice of pre-implemented tokenizer #1
        elif params.encoding_tokens == "sequence_k_tokens":
            self.tokenizer = SparseTokenizerSequenceKTokens(
                self.data_class, params.N, self.k, self.are_coordinates_symmetric, self.SPECIAL_SYMBOLS, encoding_augmentation=encoding_augmentation
            ) # <- Possible choice of pre-implemented tokenizer #2
        elif params.encoding_tokens == "adjacency":
            self.tokenizer = DenseTokenizer(
                self.data_class, params.N, self.k, self.are_coordinates_symmetric, self.SPECIAL_SYMBOLS, pow2base=params.pow2base, encoding_augmentation=encoding_augmentation
            ) # <- Possible choice of pre-implemented tokenizer #3
        else:
            raise ValueError(f"Invalid encoding: {params.encoding_tokens}")
    
    @staticmethod
    def register_args(parser):
        parser.add_argument("--N", type=int, default=10, help="Grid size N x N")
        parser.add_argument("--encoding_tokens", type=str, default="single_integer", help="single_integer/sequence_k_tokens/adjacency")
        parser.add_argument("--make_object_canonical", type=bool_flag, default="false", help="sort the grid by symmetry")
        parser.add_argument("--augment_data_representation", type=bool_flag, default="false", help="augment the data representation with predefined function")
        parser.add_argument("--pow2base", type=int, default=1, help="Bits per token for adjacency encoding")

#### Step 5: Register the Environment

Similar to the previous env, you need to add a new line in the `src/envs/__init__.py`

```
ENVS = {
    "square": SquareEnvironment,
    "isosceles": IsoscelesEnvironment,
    "sphere": SphereEnvironment,
    "collinear": CollinearEnvironment,  # Add this line
}
```